# 🩺 Diabetes Data Analysis - Step by Step Guide

Welcome to this interactive notebook where we'll analyze diabetes patient data together!

**What we'll learn:**
- How to load and explore medical data
- How to find patterns and relationships in data
- How to create beautiful visualizations
- Which health factors are most linked to diabetes

**No coding experience needed!** Just follow each step.

---

## Step 1: Setup - Import the Tools We Need

Think of this like opening a toolbox. We're getting all our analysis tools ready.

In [ ]:
# Import all the tools we'll need for this analysis
# 'pandas' - helps us work with data tables (like Excel)
import pandas as pd

# 'numpy' - helps with math calculations
import numpy as np

# 'matplotlib' and 'seaborn' - help create charts and graphs
import matplotlib.pyplot as plt
import seaborn as sns

# This line makes charts appear nicely in the notebook
%matplotlib inline

# Optional: Import our own analysis functions
from source.data_loader import load_diabetes_data, get_dataset_summary
from source.analyzer import calculate_correlations, identify_risk_factors
from source.visualizer import plot_correlation_heatmap, plot_correlation_piechart

print("✅ All tools loaded successfully!")

---

## Step 2: Load the Data

Now we'll load the diabetes dataset into our notebook. This data contains information about 768 patients.

In [ ]:
# Load the diabetes dataset
# This reads the CSV file and puts it into a 'DataFrame' (like a table)
data = pd.read_csv('diabetes.csv')

# Show the first few rows to see what our data looks like
print("📊 First 5 rows of our dataset:")
data.head()

### What does each column mean?

| Column | Meaning |
|--------|----------|
| Pregnancies | How many times the person has been pregnant |
| Glucose | Blood sugar level (higher = more sugar in blood) |
| BloodPressure | Blood pressure measurement |
| SkinThickness | Thickness of skin fold on arm |
| Insulin | Hormone that controls blood sugar |
| BMI | Body Mass Index (measure of body fat based on height/weight) |
| Age | Person's age in years |
| Outcome | 0 = No diabetes, 1 = Has diabetes |

---

## Step 3: Explore the Dataset

Let's understand our data better by looking at its shape and basic statistics.

In [ ]:
# How big is our dataset?
print(f"📈 Dataset Size: {data.shape[0]} patients, {data.shape[1]} columns")

# What types of data do we have?
print("\n📋 Data Types:")
print(data.dtypes)

# Get basic statistics for all numbers
print("\n📊 Basic Statistics (mean, std, min, max, etc.):")
data.describe()

In [ ]:
# Check how many people have diabetes vs don't
diabetes_counts = data['Outcome'].value_counts()
print("👥 Class Distribution:")
print(f"   No Diabetes (0): {diabetes_counts[0]} patients ({diabetes_counts[0]/len(data)*100:.1f}%)")
print(f"   Has Diabetes (1): {diabetes_counts[1]} patients ({diabetes_counts[1]/len(data)*100:.1f}%)")

---

## Step 4: Find Correlations - Which Factors Matter Most?

Correlation tells us how strongly two things are related. For example:
- If higher glucose → more diabetes, then glucose is **positively correlated**
- Correlation values go from -1 (opposite) to +1 (same direction)
- Values closer to 0 mean little to no relationship

In [ ]:
# Calculate correlations with diabetes outcome
# This tells us which factors are most connected to diabetes

# Remove 'Outcome' from the list since we're measuring correlation WITH it
feature_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                   'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Calculate correlation of each feature with Outcome
correlations = data[feature_columns + ['Outcome']].corr()['Outcome'].drop('Outcome')

# Sort by absolute value (strongest first)
correlations_sorted = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

print("🔗 Correlation with Diabetes (sorted by strength):")
print("-" * 50)
for feature, corr in correlations_sorted.items():
    # Create a visual bar
    bar_length = int(abs(corr) * 30)
    bar = "█" * bar_length
    sign = "+" if corr > 0 else "-"
    print(f"{feature:28s} {sign}{abs(corr):.3f} {bar}")

In [ ]:
# Show correlations as a horizontal bar chart
plt.figure(figsize=(10, 6))

# Create color based on positive/negative correlation
colors = ['#FF6B6B' if c > 0 else '#4ECDC4' for c in correlations_sorted]

# Create the bar chart
plt.barh(correlations_sorted.index, correlations_sorted.values, color=colors)

# Add labels and title
plt.xlabel('Correlation with Diabetes', fontsize=12)
plt.ylabel('Health Factor', fontsize=12)
plt.title('🔗 Which Factors are Most Connected to Diabetes?', fontsize=14, fontweight='bold')

# Add a vertical line at 0
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Add value labels on bars
for i, (idx, val) in enumerate(correlations_sorted.items()):
    plt.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 📝 What does this tell us?

**Key Findings:**
1. **Glucose (0.47)** - Strongest link! Higher blood sugar is strongly connected to diabetes
2. **BMI (0.29)** - Moderate link. Higher body weight is associated with more diabetes
3. **Age (0.24)** - Moderate link. Older people tend to have more diabetes
4. **Insulin (0.13)** - Weak link but still present
5. **BloodPressure (0.07)** - Very weak connection

---

## Step 5: Create a Correlation Heatmap

A heatmap shows how ALL factors relate to each other at once. The colors tell us:
- 🔴 Red = Strong positive correlation
- 🔵 Blue = Strong negative correlation
- ⚪ White/Light = Little to no correlation

In [ ]:
# Create a correlation heatmap
plt.figure(figsize=(14, 10))

# Calculate the full correlation matrix
correlation_matrix = data.corr()

# Create the heatmap using seaborn
sns.heatmap(
    correlation_matrix,
    annot=True,           # Show numbers in cells
    fmt='.2f',            # 2 decimal places
    cmap='RdYlBu_r',     # Red-Yellow-Blue colormap
    center=0,            # Center the colors at 0
    square=True,         # Make cells square
    linewidths=0.5,      # Add lines between cells
    cbar_kws={'shrink': 0.8}
)

plt.title('🗺️ Correlation Heatmap - All Health Factors', fontsize=16, fontweight='bold')
plt.tight_layout()

# Save the figure
plt.savefig('visualizations/Correlations with Diabetes.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/Correlations with Diabetes.png")

plt.show()

---

## Step 6: Create a Pie Chart - Feature Importance

A pie chart shows what percentage each factor contributes to predicting diabetes.

In [ ]:
# Create a pie chart showing relative importance of each factor

# Use absolute correlations for size
importance = correlations.abs().sort_values(ascending=False)

# Create figure
plt.figure(figsize=(12, 10))

# Define colors for each segment
colors = plt.cm.Set3(np.linspace(0, 1, len(importance)))

# Create pie chart
plt.pie(
    importance,
    labels=importance.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    explode=[0.05 if i == 0 else 0 for i in range(len(importance))],
    shadow=True
)

plt.title('🥧 Feature Importance - Contribution to Diabetes Prediction', 
          fontsize=14, fontweight='bold')

plt.legend(
    importance.index,
    title="Health Parameters",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()

# Save
plt.savefig('visualizations/DiabetesCorrelationPiechart.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/DiabetesCorrelationPiechart.png")

plt.show()

---

## Step 7: Analyze Each Health Factor

Let's look at how each health factor is distributed among diabetic vs non-diabetic patients.

In [ ]:
# Function to create distribution plots
def plot_factor_distribution(data, column, bins=30):
    """
    Create a histogram showing distribution of a factor
    Blue = Non-diabetic, Red = Diabetic
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Separate data by outcome
    diabetic = data[data['Outcome'] == 1][column]
    non_diabetic = data[data['Outcome'] == 0][column]
    
    # Create histograms
    ax.hist(non_diabetic, bins=bins, alpha=0.6, label='Non-Diabetic', 
            color='steelblue', edgecolor='white')
    ax.hist(diabetic, bins=bins, alpha=0.6, label='Diabetic', 
            color='indianred', edgecolor='white')
    
    # Labels
    ax.set_xlabel(column, fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Distribution of {column}\n(Blue=No Diabetes, Red=Has Diabetes)', 
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save with proper filename
    filename = f'visualizations/{column.lower()}Distribution.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"💾 Saved: {filename}")
    
    plt.show()

# Run for key factors
print("📊 Analyzing GLUCOSE (Blood Sugar Level):")
plot_factor_distribution(data, 'Glucose')

In [ ]:
print("📊 Analyzing BMI (Body Mass Index):")
plot_factor_distribution(data, 'BMI')

In [ ]:
print("📊 Analyzing AGE:")
plot_factor_distribution(data, 'Age')

In [ ]:
print("📊 Analyzing INSULIN Level:")
plot_factor_distribution(data, 'Insulin')

In [ ]:
print("📊 Analyzing BLOOD PRESSURE:")
plot_factor_distribution(data, 'BloodPressure')

---

## Step 8: Grouped Analysis - Age and BMI Categories

Let's see how diabetes rates change across different age groups and BMI categories.

In [ ]:
# Create age groups
data_copy = data.copy()
data_copy['AgeGroup'] = pd.cut(
    data_copy['Age'], 
    bins=[20, 30, 40, 50, 60, 70, 80, 90],
    labels=['20-30', '30-40', '40-50', '50-60', '60-70', '70-80', '80-90']
)

# Create bar chart
fig, ax = plt.subplots(figsize=(12, 7))

# Group by age and count outcomes
age_counts = data_copy.groupby(['AgeGroup', 'Outcome']).size().unstack(fill_value=0)

# Plot bars
x = np.arange(len(age_counts.index))
width = 0.35

bars1 = ax.bar(x - width/2, age_counts[0], width, label='Non-Diabetic', color='steelblue')
bars2 = ax.bar(x + width/2, age_counts[1], width, label='Diabetic', color='indianred')

# Labels
plt.xlabel('Age Group (years)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Patients', fontsize=12)
plt.title('📊 Diabetes Distribution Across Age Groups', fontsize=14, fontweight='bold')
plt.xticks(x, age_counts.index, rotation=45)
plt.legend(fontsize=11)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/Diabetes Distribution Across Age Groups.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/Diabetes Distribution Across Age Groups.png")
plt.show()

In [ ]:
# Create BMI groups (using medical categories)
# Underweight: <18.5, Normal: 18.5-24.9, Overweight: 25-29.9, Obese: 30+
data_copy['BMIGroup'] = pd.cut(
    data_copy['BMI'],
    bins=[0, 18.5, 24.9, 29.9, 34.9, 40, 50],
    labels=['<18.5 (Underweight)', '18.5-25 (Normal)', '25-30 (Overweight)', 
            '30-35 (Obese)', '35-40 (Very Obese)', '40+ (Extremely Obese)']
)

# Create bar chart
fig, ax = plt.subplots(figsize=(14, 7))

# Group by BMI and count outcomes
bmi_counts = data_copy.groupby(['BMIGroup', 'Outcome']).size().unstack(fill_value=0)

# Plot bars
x = np.arange(len(bmi_counts.index))
width = 0.35

bars1 = ax.bar(x - width/2, bmi_counts[0], width, label='Non-Diabetic', color='steelblue')
bars2 = ax.bar(x + width/2, bmi_counts[1], width, label='Diabetic', color='indianred')

# Labels
plt.xlabel('BMI Category', fontsize=12, fontweight='bold')
plt.ylabel('Number of Patients', fontsize=12)
plt.title('📊 Diabetes Distribution Across BMI Levels', fontsize=14, fontweight='bold')
plt.xticks(x, bmi_counts.index, rotation=45, ha='right')
plt.legend(fontsize=11)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/Diabetes Distribution Across BMI Levels.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/Diabetes Distribution Across BMI Levels.png")
plt.show()

In [ ]:
# Create Glucose level groups
# Normal: <100, Prediabetic: 100-125, Diabetic: 126+
data_copy['GlucoseGroup'] = pd.cut(
    data_copy['Glucose'],
    bins=[0, 70, 100, 140, 180, 250, 300],
    labels=['0-70', '70-100', '100-140', '140-180', '180-250', '250-300']
)

# Create bar chart
fig, ax = plt.subplots(figsize=(12, 7))

# Group by glucose and count outcomes
glucose_counts = data_copy.groupby(['GlucoseGroup', 'Outcome']).size().unstack(fill_value=0)

# Plot bars
x = np.arange(len(glucose_counts.index))
width = 0.35

bars1 = ax.bar(x - width/2, glucose_counts[0], width, label='Non-Diabetic', color='steelblue')
bars2 = ax.bar(x + width/2, glucose_counts[1], width, label='Diabetic', color='indianred')

# Labels
plt.xlabel('Glucose Level Range (mg/dL)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Patients', fontsize=12)
plt.title('📊 Diabetes Distribution Across Glucose Levels', fontsize=14, fontweight='bold')
plt.xticks(x, glucose_counts.index, rotation=45)
plt.legend(fontsize=11)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/Diabetes Distribution Across Glucose Levels.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/Diabetes Distribution Across Glucose Levels.png")
plt.show()

In [ ]:
# Create Insulin level groups
data_copy['InsulinGroup'] = pd.cut(
    data_copy['Insulin'],
    bins=[0, 50, 100, 150, 200, 250, 300],
    labels=['0-50', '50-100', '100-150', '150-200', '200-250', '250-300']
)

# Create bar chart
fig, ax = plt.subplots(figsize=(12, 7))

# Group by insulin and count outcomes
insulin_counts = data_copy.groupby(['InsulinGroup', 'Outcome']).size().unstack(fill_value=0)

# Plot bars
x = np.arange(len(insulin_counts.index))
width = 0.35

bars1 = ax.bar(x - width/2, insulin_counts[0], width, label='Non-Diabetic', color='steelblue')
bars2 = ax.bar(x + width/2, insulin_counts[1], width, label='Diabetic', color='indianred')

# Labels
plt.xlabel('Insulin Level Range (mu U/ml)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Patients', fontsize=12)
plt.title('📊 Diabetes Distribution Across Insulin Levels', fontsize=14, fontweight='bold')
plt.xticks(x, insulin_counts.index, rotation=45)
plt.legend(fontsize=11)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/Diabetes Distribution Across Insulin Levels.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/Diabetes Distribution Across Insulin Levels.png")
plt.show()

In [ ]:
# Create Blood Pressure level groups
data_copy['BPGroup'] = pd.cut(
    data_copy['BloodPressure'],
    bins=[0, 80, 100, 120, 140, 160, 200],
    labels=['<80 (Low)', '80-100 (Normal)', '100-120 (Elevated)', 
            '120-140 (High Stage 1)', '140-160 (High Stage 2)', '160+ (Crisis)']
)

# Create bar chart
fig, ax = plt.subplots(figsize=(14, 7))

# Group by BP and count outcomes
bp_counts = data_copy.groupby(['BPGroup', 'Outcome']).size().unstack(fill_value=0)

# Plot bars
x = np.arange(len(bp_counts.index))
width = 0.35

bars1 = ax.bar(x - width/2, bp_counts[0], width, label='Non-Diabetic', color='steelblue')
bars2 = ax.bar(x + width/2, bp_counts[1], width, label='Diabetic', color='indianred')

# Labels
plt.xlabel('Blood Pressure Level (mm Hg)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Patients', fontsize=12)
plt.title('📊 Diabetes Distribution Across Blood Pressure Levels', fontsize=14, fontweight='bold')
plt.xticks(x, bp_counts.index, rotation=45, ha='right')
plt.legend(fontsize=11)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/Diabetes Distribution Across Blood Pressure Levels.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/Diabetes Distribution Across Blood Pressure Levels.png")
plt.show()

---

## Step 9: Create a Pairplot - Relationships Between Variables

A pairplot shows scatter plots of every variable pair. This helps us see patterns like:
- Do higher glucose levels lead to higher insulin?
- Is there a relationship between BMI and Age?

In [ ]:
# Select key variables for pairplot
# (Pairplot can be slow with too many variables, so we select the most important ones)
key_vars = ['Glucose', 'BMI', 'Age', 'Insulin', 'Outcome']

print("Creating pairplot... (this may take a moment)")

# Create pairplot
g = sns.pairplot(
    data[key_vars],
    hue='Outcome',
    palette={0: 'steelblue', 1: 'indianred'},
    diag_kind='hist',
    plot_kws={'alpha': 0.6, 's': 50},
    height=2.5
)

# Add title
g.figure.suptitle('🔗 Relationships Between Key Health Variables', y=1.02, fontsize=14, fontweight='bold')

# Save
g.savefig('visualizations/pairplot.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/pairplot.png")

plt.show()

---

## Step 10: Summary Statistics - Compare Groups

Let's calculate and compare the average values of each factor between diabetic and non-diabetic patients.

In [ ]:
# Separate data by outcome
diabetic = data[data['Outcome'] == 1]
non_diabetic = data[data['Outcome'] == 0]

# Create comparison table
comparison_data = []

for column in feature_columns:
    comparison_data.append({
        'Factor': column,
        'Non-Diabetic Mean': non_diabetic[column].mean(),
        'Diabetic Mean': diabetic[column].mean(),
        'Difference': diabetic[column].mean() - non_diabetic[column].mean(),
        '% Difference': ((diabetic[column].mean() - non_diabetic[column].mean()) / non_diabetic[column].mean() * 100)
    })

comparison_df = pd.DataFrame(comparison_data)

# Format for better display
print("📊 COMPARISON: Diabetic vs Non-Diabetic Patients")
print("=" * 80)
comparison_df['Non-Diabetic Mean'] = comparison_df['Non-Diabetic Mean'].round(2)
comparison_df['Diabetic Mean'] = comparison_df['Diabetic Mean'].round(2)
comparison_df['Difference'] = comparison_df['Difference'].round(2)
comparison_df['% Difference'] = comparison_df['% Difference'].round(1)

print(comparison_df.to_string(index=False))

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(14, 8))

# Create grouped bar chart
factors = comparison_df['Factor'].values
non_diab_means = comparison_df['Non-Diabetic Mean'].values
diab_means = comparison_df['Diabetic Mean'].values

x = np.arange(len(factors))
width = 0.35

bars1 = ax.bar(x - width/2, non_diab_means, width, label='Non-Diabetic', color='steelblue')
bars2 = ax.bar(x + width/2, diab_means, width, label='Diabetic', color='indianred')

# Labels
plt.xlabel('Health Factor', fontsize=12, fontweight='bold')
plt.ylabel('Average Value', fontsize=12)
plt.title('📊 Comparison of Health Factors: Diabetic vs Non-Diabetic', fontsize=14, fontweight='bold')
plt.xticks(x, factors, rotation=45, ha='right')
plt.legend(fontsize=11)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8, rotation=90)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8, rotation=90)

plt.tight_layout()
plt.savefig('visualizations/GroupComparison.png', dpi=300, bbox_inches='tight')
print("💾 Saved: visualizations/GroupComparison.png")
plt.show()

---

## 🎉 Final Summary

### Key Findings from the Analysis:

#### 1. **Glucose (Blood Sugar) is the Strongest Predictor**
- Correlation: 0.47 (moderate to strong)
- Diabetic patients have 22% higher glucose on average
- This makes sense medically - diabetes is a blood sugar disease

#### 2. **BMI (Body Weight) Matters**
- Correlation: 0.29 (moderate)
- Diabetic patients have 13% higher BMI
- Higher body weight is associated with more diabetes risk

#### 3. **Age is a Factor**
- Correlation: 0.24 (moderate)
- Older people tend to have more diabetes
- Risk increases significantly after age 40

#### 4. **Other Factors**
- Insulin: Weak correlation (0.13)
- Pregnancies: Weak correlation (0.22)
- BloodPressure: Very weak correlation (0.07)

---

### What Charts Did We Create?

| Chart | Description |
|-------|-------------|
| Correlation Heatmap | Shows how all factors relate to each other |
| Pie Chart | Shows which factors matter most for diabetes |
| Glucose Distribution | Shows blood sugar patterns |
| BMI Distribution | Shows weight patterns |
| Age Group Bar Chart | Shows diabetes by age ranges |
| BMI Group Bar Chart | Shows diabetes by weight categories |
| Glucose Level Bar Chart | Shows diabetes by blood sugar ranges |
| Insulin Level Bar Chart | Shows diabetes by insulin levels |
| Blood Pressure Bar Chart | Shows diabetes by BP ranges |
| Pairplot | Shows all variable relationships |
| Comparison Chart | Compares diabetic vs non-diabetic averages |

### All visualizations saved to: `visualizations/` folder

---

## 🔧 How to Save This Notebook

1. Click **File** → **Download as** → **Notebook (.ipynb)**
2. Or click **File** → **Download as** → **HTML** to view in a web browser

### To Run Again Later:
```bash
# Activate your virtual environment
.\venv\Scripts\activate

# Start Jupyter notebook
jupyter notebook

# Navigate to this notebook and run all cells
```